In [1]:
!pip install pandas catboost numpy lightgbm rectools-lightfm tqdm -q

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.preprocessing import LabelEncoder

In [2]:

import pandas as pd
from more_itertools import pairwise
import numpy as np

class TimeRangeSplit():
    """
        https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.date_range.html
    """
    def __init__(self, 
                 start_date, 
                 end_date=None,
                 freq='D',
                 periods=None, 
                 tz=None, 
                 normalize=False, 
                 closed=None, 
                 train_min_date=None,
                 filter_cold_users=True, 
                 filter_cold_items=True, 
                 filter_already_seen=True):
        
        self.start_date = start_date
        if end_date is None and periods is None:
            raise ValueError("Either 'end_date' or 'periods' must be non-zero, not both at the same time.")

        self.end_date = end_date
        self.freq = freq
        self.periods = periods
        self.tz = tz
        self.normalize = normalize
        self.closed = closed
        self.train_min_date = pd.to_datetime(train_min_date, errors='raise')
        self.filter_cold_users = filter_cold_users
        self.filter_cold_items = filter_cold_items
        self.filter_already_seen = filter_already_seen

        self.date_range = pd.date_range(
            start=start_date, 
            end=end_date, 
            freq=freq, 
            periods=periods, 
            tz=tz, 
            normalize=normalize, 
            )

        self.max_n_splits = max(0, len(self.date_range) - 1)
        if self.max_n_splits == 0:
            raise ValueError("Provided parametrs set an empty date range.") 

    def split(self, 
              df, 
              user_column='user_id',
              item_column='item_id',
              datetime_column='date',
              fold_stats=False):
        df_datetime = df[datetime_column]
        if self.train_min_date is not None:
            train_min_mask = df_datetime >= self.train_min_date
        else:
            train_min_mask = df_datetime.notnull()

        date_range = self.date_range[(self.date_range >= df_datetime.min()) & 
                                     (self.date_range <= df_datetime.max())]

        for start, end in pairwise(date_range):
            fold_info = {
                'Start date': start,
                'End date': end
            }
            train_mask = train_min_mask & (df_datetime < start)
            train_idx = df.index[train_mask]
            if fold_stats:
                fold_info['Train'] = len(train_idx)

            test_mask = (df_datetime >= start) & (df_datetime < end)
            test_idx = df.index[test_mask]
            
            if self.filter_cold_users:
                new = np.setdiff1d(
                    df.loc[test_idx, user_column].unique(), 
                    df.loc[train_idx, user_column].unique())
                new_idx = df.index[test_mask & df[user_column].isin(new)]
                test_idx = np.setdiff1d(test_idx, new_idx)
                test_mask = df.index.isin(test_idx)
                if fold_stats:
                    fold_info['New users'] = len(new)
                    fold_info['New users interactions'] = len(new_idx)

            if self.filter_cold_items:
                new = np.setdiff1d(
                    df.loc[test_idx, item_column].unique(), 
                    df.loc[train_idx, item_column].unique())
                new_idx = df.index[test_mask & df[item_column].isin(new)]
                test_idx = np.setdiff1d(test_idx, new_idx)
                test_mask = df.index.isin(test_idx)
                if fold_stats:
                    fold_info['New items'] = len(new)
                    fold_info['New items interactions'] = len(new_idx)

            if self.filter_already_seen:
                user_item = [user_column, item_column]
                train_pairs = df.loc[train_idx, user_item].set_index(user_item).index
                test_pairs = df.loc[test_idx, user_item].set_index(user_item).index
                intersection = train_pairs.intersection(test_pairs)
                test_idx = test_idx[~test_pairs.isin(intersection)]
                # test_mask = rd.df.index.isin(test_idx)
                if fold_stats:
                    fold_info['Known interactions'] = len(intersection)

            if fold_stats:
                fold_info['Test'] = len(test_idx)

            yield (train_idx, test_idx, fold_info)

    def get_n_splits(self, df, datetime_column='date'):
        df_datetime = df[datetime_column]
        if self.train_min_date is not None:
            df_datetime = df_datetime[df_datetime >= self.train_min_date]

        date_range = self.date_range[(self.date_range >= df_datetime.min()) & 
                                     (self.date_range <= df_datetime.max())]

        return max(0, len(date_range) - 1)

In [23]:
base_dir = Path('.')
train = pd.read_csv(base_dir / 'train.csv')
owner = pd.read_csv(base_dir / 'owner.csv')
sample_sub = pd.read_csv(base_dir / 'sample_submission.csv')
user = pd.read_csv(base_dir / 'user.csv', parse_dates=['create_date'])
video = pd.read_csv(base_dir / 'video.csv')

In [24]:
owner['create_date'] = pd.to_datetime(owner['create_date'], unit='ms')
owner['last_active_date'] = pd.to_datetime(owner['last_active_date'], unit='ms')
video['upload_timestamp'] = pd.to_datetime(video['upload_timestamp'], unit='ms')
train['timestamp'] = pd.to_datetime(train['timestamp'], unit='ms')

In [25]:
user['gender'] = user['gender'].apply(lambda x: 1 if x == 'F' else 0)

enc = LabelEncoder()
user['language'] = enc.fit_transform(user['language'].values)
user

,user_id,gender,age,language,city_id,birth_city_id,create_date
0,938,1,57.0,22,5f7ca800fcb9368f78e3740cb68a4c4ebc62b005cd15cf...,NaN,2011-03-11 21:00:00
1,57571,1,68.0,22,NaN,NaN,2011-03-11 21:00:00
2,50873,1,66.0,22,a26a3a5b73942ee4af156df272ba2e722ddf4eb50ea396...,NaN,2011-03-11 21:00:00
3,4335,1,66.0,22,5f7ca800fcb9368f78e3740cb68a4c4ebc62b005cd15cf...,NaN,2011-03-11 21:00:00
4,42138,1,66.0,22,e1baf026d2d5c938c8ca66bac84c655345b92333a74937...,6b415eabf81e3cc85adac7d323a989a92198e76fe3f053...,2011-03-11 21:00:00
...,...,...,...,...,...,...,...
152906,79591,0,17.0,22,NaN,NaN,2022-10-10 21:00:00
152907,93648,1,101.0,22,NaN,NaN,2022-10-10 21:00:00
152908,42337,0,37.0,22,NaN,NaN,2022-10-10 21:00:00
152909,4694,1,78.0,22,cc15ba5ee579964ee4d5b88d1b7d6e31ca5b3ddda47366...,cc15ba5ee579964ee4d5b88d1b7d6e31ca5b3ddda47366...,2022-10-10 21:00:00


In [26]:
train['interaction_type'] = train['interaction_type'].apply(lambda x: 3 if x == 'like' else 1)

## train
    |
    |- user id
    |- item id
    |- interaction
    |- time

---

## user
    |
    |- user id
    |- gender
    |- age
    |- lang
    |- city id
    |- birth city id
    |- create account date

---

## video
    |
    |- video id
    |- owner id
    |- duration
    |- upload timestamp

---

## owner
    |
    |- owner id
    |- subscribers count
    |- last active
    |- city id
    |- account create date

In [27]:
last_date = train['timestamp'].max()
first_date = train['timestamp'].min()
folds = 7
start_date = last_date - pd.Timedelta(days=folds)
start_date, last_date

(Timestamp('2023-10-24 20:43:51.021000'),
 Timestamp('2023-10-31 20:43:51.021000'))

In [28]:
cv = TimeRangeSplit(
    start_date=start_date,
    periods=folds+1, 
    train_min_date = first_date
)

folds_with_stats = list(cv.split(
    train,
    user_column='user_id',
    item_column='video_id',
    datetime_column='timestamp',
    fold_stats=True
))

folds_info_with_stats = pd.DataFrame([info for _, _, info in folds_with_stats])
folds_info_with_stats

,Start date,End date,Train,New users,New users interactions,New items,New items interactions,Known interactions,Test
0,2023-10-24 20:43:51.021,2023-10-25 20:43:51.021,5094578,22,369,4667,29343,2152,129554
1,2023-10-25 20:43:51.021,2023-10-26 20:43:51.021,5256192,23,282,4799,23925,1886,108506
2,2023-10-26 20:43:51.021,2023-10-27 20:43:51.021,5391020,15,269,4926,17077,1583,83164
3,2023-10-27 20:43:51.021,2023-10-28 20:43:51.021,5493323,19,257,4322,13233,1201,58703
4,2023-10-28 20:43:51.021,2023-10-29 20:43:51.021,5566959,12,241,3426,9152,857,40993
5,2023-10-29 20:43:51.021,2023-10-30 20:43:51.021,5618412,2,27,1696,4551,505,21037
6,2023-10-30 20:43:51.021,2023-10-31 20:43:51.021,5644661,5,165,1077,1718,236,10465


# Base line with CatBoostRanker & LightGBM blend

### Candidates generation with LightFM

In [29]:
from lightfm import LightFM
from lightfm.data import Dataset
from sklearn.model_selection import TimeSeriesSplit

dataset = Dataset()

/Users/ilagulakin/Desktop/AI & code/RecSys/NTO_BDML_2024/.venv/lib/python3.14/site-packages/lightfm/_lightfm_fast.py:9: UserWarning: LightFM was compiled without OpenMP support. Only a single thread will be used.
  warnings.warn(


In [30]:
def time_features(data, cols):
    for col in cols:
        data[f'month_cos_{col}'] = np.cos(data[col].dt.month * 2 * np.pi / 12)
        data[f'month_sin_{col}'] = np.sin(data[col].dt.month * 2 * np.pi / 12)
        data[f'day_cos_{col}'] = np.cos(data[col].dt.day * 2 * np.pi / 31)
        data[f'day_sin_{col}'] = np.sin(data[col].dt.day * 2 * np.pi / 31)
        data[f'hour_cos_{col}'] = np.cos(data[col].dt.hour * 2 * np.pi / 24)
        data[f'hour_sin_{col}'] = np.sin(data[col].dt.hour * 2 * np.pi / 24)
        data[f'minute_cos_{col}'] = np.cos(data[col].dt.minute * 2 * np.pi / 60)
        data[f'minute_sin_{col}'] = np.sin(data[col].dt.minute * 2 * np.pi / 60)

    return data

time_cols = ['month_cos', 'month_sin', 'day_cos', 'day_sin', 'hour_cos', 'hour_sin', 'minute_cos', 'minute_sin']
owner = time_features(owner, ['last_active_date', 'create_date'])
user = time_features(user, ['create_date'])
video = time_features(video, ['upload_timestamp'])

In [31]:
train

,interaction_type,timestamp,user_id,video_id
0,1,2023-10-02 18:24:39.748,126492,228525
1,1,2023-10-02 18:24:39.760,117764,204343
2,1,2023-10-02 18:24:39.988,11347,201337
3,1,2023-10-02 18:24:40.451,125274,221842
4,1,2023-10-02 18:24:40.451,125274,221842
...,...,...,...,...
5657319,1,2023-10-10 14:00:05.821,20627,42064
5657320,1,2023-10-10 14:00:06.884,73913,242899
5657321,1,2023-10-10 14:00:07.292,53555,241941
5657322,1,2023-10-10 14:00:07.461,97851,222069


In [32]:
dataset.fit(train['user_id'].unique(), train['video_id'].unique())

In [33]:
owner['city_id'] = owner['city_id'].astype('category')
owner['city_id'] = owner['city_id'].cat.add_categories('unknown_city')
owner['city_id'] = owner['city_id'].fillna('unknown_city')
owner.city_id.unique()

['unknown_city', '8fb6fe36d1084805b2d52275f0de118124693863427a6..., '3fc0d5d7ab6cdbbf96dc9c648a930dc31be7e8a50db54..., 'e0abcde3c9a0bdc6d3b2acc51730a07e5655d7df6a7bd..., '9dcd5aac328b31b2b473ee3b4fc3d5245bf1c516bd6fc..., ..., 'eabb8ed7a8fb7e308f27374c7607496da862ce08cfec3..., '47e48bba1839edb803c2a2c14943788592452b0bc984a..., '3c1c76dffd287ea315f73779a63fa4b91c46c3daa55f0..., '072d0abd67d22cee67d67df51adb001c0c48df095075f..., '6a09d859abe5c7bde0616a58ca52cbdb2f173077e4a14...]
Length: 575
Categories (575, str): ['00011c2890d9a0d12362aedae266135ec8dbc2b4876c9..., '00b317bb677a44b1f69e6bd18415b3da6680a0844f55e..., '0113159d9a667967a86607e301d8269c3c56c42929b45..., '015e9e30d71d70fcc6d7cd592928b0cebcfa28394d3be..., ..., 'ff5e54a4db122cb412691ec3fa7226131c38b767d6804..., 'ffcbf81621504d8631d200f0850719a0bbb2b9e7fb12e..., 'ffd041ea266da01e17a4f99a7a44aacef430a45408968..., 'unknown_city']

In [34]:
user['city_id'] = user['city_id'].astype('category')
user['city_id'] = user['city_id'].cat.add_categories('unknown_city')
user['city_id'] = user['city_id'].fillna('unknown_city')

user['birth_city_id'] = user['birth_city_id'].astype('category')
user['birth_city_id'] = user['birth_city_id'].cat.add_categories('unknown_city')
user['birth_city_id'] = user['birth_city_id'].fillna('unknown_city')

user['age'] = user['age'].fillna(-1)


In [35]:
items = video.merge(owner, on='owner_id', how='left')
users = user

In [36]:
def get_features(data: pd.DataFrame, drop: str) -> list[str]:
    features_names = []
    for feature_col in [col for col in data.columns if col != drop]:
        vals = data[feature_col].unique()
        for val in vals: 
            features_names.append(f'{feature_col}:{val}')
    
    return features_names

user_features_names = get_features(users, 'user_id')
item_features_names = get_features(items, 'video_id')

In [37]:
dataset.fit_partial(user_features=user_features_names)
dataset.fit_partial(item_features=item_features_names)

In [38]:
lightfm_mapping = dataset.mapping()
lightfm_mapping = {
    'user_mapping': lightfm_mapping[0],
    'user_features_mapping': lightfm_mapping[1],
    'item_mapping': lightfm_mapping[2],
    'item_features_mapping': lightfm_mapping[3],
}

lightfm_mapping['user_inv_mapping'] = {v: k for k, v in lightfm_mapping['user_mapping'].items()}
lightfm_mapping['item_inv_mapping'] = {v: k for k, v in lightfm_mapping['item_mapping'].items()}

In [39]:
def df_to_tuple_iterator(df):
    return zip(*df.values.T)

def concat_last_to_list(t):
    return (t[0], list(t[1: ])[0])

def df_to_tuple_list_iterator(df):
    return map(concat_last_to_list, df_to_tuple_iterator(df))

In [40]:
train_idx, test_idx, info = folds_with_stats[0]

tr_split = train.loc[train_idx]
te_split = train.loc[test_idx]

In [41]:
train_mat, train_mat_weights = dataset.build_interactions(df_to_tuple_iterator(train[['user_id', 'video_id', 'interaction_type']]))
train_mat

<COOrdinate sparse matrix of dtype 'int32'
	with 5657324 stored elements and shape (152911, 228506)>

In [42]:
user_types_map = {
    'gender': int,
    'age': int, 
    'language': int,
    'city_id': str,
    'birth_city_id': str,
    'month_cos_create_date': float,
    'month_sin_create_date': float,
    'day_cos_create_date': float,
    'day_sin_create_date': float,
    'hour_cos_create_date': float,
    'hour_sin_create_date': float,
    'minute_cos_create_date': float,
    'minute_sin_create_date': float
}

user_features = ['gender', 'age', 'language', 'city_id', 'birth_city_id',
                 'month_cos_create_date', 'month_sin_create_date', 'day_cos_create_date',
                 'day_sin_create_date', 'hour_cos_create_date', 'hour_sin_create_date',
                 'minute_cos_create_date', 'minute_sin_create_date'
                 ]

items_features = [
                'owner_id', 'duration', 
                'month_cos_upload_timestamp', 'month_sin_upload_timestamp',
                'day_cos_upload_timestamp', 'day_sin_upload_timestamp',
                'hour_cos_upload_timestamp', 'hour_sin_upload_timestamp',
                'minute_cos_upload_timestamp', 'minute_sin_upload_timestamp',
                'subscribers_count', 'city_id',
                'month_cos_last_active_date', 'month_sin_last_active_date',
                'day_cos_last_active_date', 'day_sin_last_active_date',
                'hour_cos_last_active_date', 'hour_sin_last_active_date',
                'minute_cos_last_active_date', 'minute_sin_last_active_date',
                'month_cos_create_date', 'month_sin_create_date', 'day_cos_create_date',
                'day_sin_create_date', 'hour_cos_create_date', 'hour_sin_create_date',
                'minute_cos_create_date', 'minute_sin_create_date'
]

item_types_map = {
    'owner_id': object, 
    'duration': int, 
    'month_cos_upload_timestamp': float, 
    'month_sin_upload_timestamp': float,
    'day_cos_upload_timestamp': float,
    'day_sin_upload_timestamp': float,
    'hour_cos_upload_timestamp': float,
    'hour_sin_upload_timestamp': float,
    'minute_cos_upload_timestamp': float,
    'minute_sin_upload_timestamp': float,
    'subscribers_count': int,
    'city_id': str,
    'month_cos_last_active_date': float,
    'month_sin_last_active_date': float,
    'day_cos_last_active_date': float,
    'day_sin_last_active_date': float,
    'hour_cos_last_active_date': float,
    'hour_sin_last_active_date': float,
    'minute_cos_last_active_date': float,
    'minute_sin_last_active_date': float,
    'month_cos_create_date': float,
    'month_sin_create_date': float,
    'day_cos_create_date': float,
    'day_sin_create_date': float,
    'hour_cos_create_date': float,
    'hour_sin_create_date': float,
    'minute_cos_create_date': float,
    'minute_sin_create_date': float
}

In [43]:
users['features'] = users[user_features].apply(
    lambda row: [f"{col}:{row[col]}" for col in user_features], axis=1
)

items['features'] = items[items_features].apply(
    lambda row: [f"{col}:{row[col]}" for col in items_features], axis=1
)

In [24]:
known_users_filter = users['user_id'].isin(train['user_id'].unique())
train_user_features = dataset.build_user_features(
    df_to_tuple_iterator(
        users.loc[known_users_filter, ['user_id', 'features']]
    )
)

known_items_filter = items['video_id'].isin(train['video_id'].unique())
train_items_features = dataset.build_item_features(
    df_to_tuple_iterator(
        items.loc[known_items_filter, ['video_id', 'features']]
    )
)


In [25]:
lfm_model = LightFM(
    no_components=64,
    learning_rate=0.05, 
    loss='warp',
    max_sampled=5,
    random_state=42
)

In [26]:
from tqdm.auto import tqdm

num_epochs = 7
for _ in tqdm(range(num_epochs), total=num_epochs):
    lfm_model.fit_partial(
        train_mat, 
        user_features=train_user_features,
        item_features=train_items_features,
        num_threads=4
    )

  0%|          | 0/7 [00:00<?, ?it/s]

In [28]:
import pickle

# Сохранение модели в файл
with open('lightfm_model.pkl', 'wb') as f:
    pickle.dump(lfm_model, f, protocol=pickle.HIGHEST_PROTOCOL)

In [ ]:
import sys
%%time

# time == 771m 40.5 s
def generate_lightfm_recs_batch(model, user_ids, item_ids, known_items,
                                 N, user_features, item_features,
                                 user_mapping, item_inv_mapping, num_threads=4):
    # Конвертируем item_ids в np.array ОДИН раз (главный источник тормозов раньше)
    item_ids_arr = np.array(item_ids, dtype=np.int32)
    n_items = len(item_ids_arr)
    
    result = {}
    total = len(user_ids)
    
    for i, uid in enumerate(user_ids):
        # Прогресс-бар через print — работает всегда
        if i % 500 == 0:
            print(f"\r[{i}/{total}] ({100*i/total:.1f}%)", end="", flush=True)
        
        if uid not in user_mapping:
            continue
        
        int_uid = user_mapping[uid]
        user_ids_repeated = np.full(n_items, int_uid, dtype=np.int32)
        
        scores = model.predict(
            user_ids_repeated,
            item_ids_arr,
            user_features=user_features,
            item_features=item_features,
            num_threads=num_threads
        )
        
        # Зануляем known items
        user_known = known_items.get(uid, [])
        if user_known:
            for vid in user_known:
                iid = lightfm_mapping['item_mapping'].get(vid)
                if iid is not None:
                    scores[iid] = -np.inf
        
        # argpartition быстрее полной сортировки
        top_idx = np.argpartition(-scores, N)[:N]
        top_idx = top_idx[np.argsort(-scores[top_idx])]
        
        recs = [item_inv_mapping[idx] for idx in top_idx if idx in item_inv_mapping]
        result[uid] = recs
    
    print(f"\r[{total}/{total}] (100.0%) — Done!")
    return result

known_items = train.groupby('user_id')['video_id'].apply(list).to_dict()
all_item_ids = list(lightfm_mapping['item_mapping'].values())

mapper = generate_lightfm_recs_batch(
    lfm_model,
    user_ids=candidates['user_id'].values,
    item_ids=all_item_ids,
    known_items=known_items,
    N=100,
    user_features=train_user_features,
    item_features=train_items_features,
    user_mapping=lightfm_mapping['user_mapping'],
    item_inv_mapping=lightfm_mapping['item_inv_mapping'],
    num_threads=4
)

[152911/152911] (100.0%) — Done!


In [ ]:
import pickle

%%time
with open('mapper.pkl', 'wb') as f:
    pickle.dump(mapper, f, protocol=pickle.HIGHEST_PROTOCOL)
print("Saved!")

Saved!


In [7]:
import pickle

with open('mapper.pkl', 'rb') as f:
    mapper = pickle.load(f)
print('Loaded')

Loaded


In [8]:
candidates = pd.DataFrame({
    'user_id': sample_sub.user_id.unique()
})

In [9]:
candidates['candidates'] = candidates['user_id'].map(mapper)
candidates

,user_id,candidates
0,938,"[188589, 51799, 34204, 13009, 131005, 89274, 1..."
1,57571,"[188589, 34204, 132648, 45311, 166825, 123953,..."
2,50873,"[188589, 51799, 34204, 132648, 131005, 123953,..."
3,4335,"[188589, 51799, 34204, 131005, 13009, 132648, ..."
4,42138,"[188589, 51799, 34204, 131005, 132648, 13009, ..."
...,...,...
152906,79591,"[236282, 184336, 216455, 238698, 11250, 166825..."
152907,93648,"[188589, 34204, 132648, 131005, 166825, 51799,..."
152908,42337,"[236282, 216455, 184336, 238698, 11250, 233131..."
152909,4694,"[188589, 34204, 51799, 131005, 132648, 45311, ..."


**Ranking**

In [10]:
from catboost import Pool, CatBoostRanker
import lightgbm as lgb
from tqdm.auto import tqdm
import gc
import matplotlib.pyplot as plt

In [ ]:
def expand_candidates(candidates):
    candidates_new = []
    for idx, row in tqdm(candidates.iterrows(), total=len(candidates.user_id.unique())):
        uid = row.user_id
        candidates = row.candidates
        for cand in candidates:
            candidates_new.append({'user_id': uid, 'video_id': cand})

    return pd.DataFrame(candidates_new)

In [12]:
candidates = expand_candidates(candidates)

  0%|          | 0/152911 [00:00<?, ?it/s]

prepare data for validation

In [44]:
max_date = train.timestamp.max()
split_date = max_date - pd.Timedelta(days=7)
train_part = train[train.timestamp < split_date]
val_part = train[train.timestamp >= split_date]

In [50]:
def generate_ranker_data(interactions_df, items_df, candidates_df=None, neg_per_pos=3, max_neg=30):
    user_positives = interactions_df.groupby('user_id')['video_id'].apply(set).to_dict()
    all_editions = set(items_df['video_id'].unique())
    
    rows = []
    
    for uid in tqdm(interactions_df['user_id'].unique()):
        positives = user_positives.get(uid, set())
        
        # Positives: relevance = 3
        for eid in positives:   
            rows.append({'user_id': uid, 'video_id': eid, 'relevance': 3})
        
        # Negatives: sample from non-interacted items
        if candidates_df is not None:
            user_cands = set(candidates_df[candidates_df['user_id'] == uid]['video_id'])
            neg_pool = user_cands - positives
        else:
            neg_pool = all_editions - positives
        
        n_neg = min(max_neg, len(positives) * neg_per_pos, len(neg_pool))
        
        if n_neg > 0 and len(neg_pool) > 0:
            negatives = np.random.choice(list(neg_pool), size=n_neg, replace=False)
            for eid in negatives:
                rows.append({'user_id': uid, 'video_id': eid, 'relevance': 0})
    
    return pd.DataFrame(rows)

In [57]:
train_df = generate_ranker_data(train_part, items, candidates)
val_df = generate_ranker_data(val_part, items, candidates)

  0%|          | 0/152813 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
def add_features(df):
    df = df.merge(items.drop(columns=['upload_timestamp', 'last_active_date', 'create_date', 'features']), on='video_id', how='left')
    df = df.merge(users.drop(columns=['create_date', 'features']), on='user_id', how='left')
    return df

In [76]:
train_df = add_features(candidates)
val_df = add_features(candidates)


cat_cols = ['owner_id', 'city_id_x', 'city_id_y', 'gender', 'language', 'birth_city_id']
feature_cols = [col for col in train_df if col not in ['relevance', 'user_id', 'video_id']]

In [77]:
train_df.head()

,user_id,video_id,owner_id,duration,month_cos_upload_timestamp,month_sin_upload_timestamp,day_cos_upload_timestamp,day_sin_upload_timestamp,hour_cos_upload_timestamp,hour_sin_upload_timestamp,...,city_id_y,birth_city_id,month_cos_create_date_y,month_sin_create_date_y,day_cos_create_date_y,day_sin_create_date_y,hour_cos_create_date_y,hour_sin_create_date_y,minute_cos_create_date_y,minute_sin_create_date_y
0,938,188589,5754,57,5.000000e-01,-0.866025,0.688967,0.724793,-0.258819,0.965926,...,5f7ca800fcb9368f78e3740cb68a4c4ebc62b005cd15cf...,unknown_city,6.123234e-17,1.0,-0.612106,0.790776,0.707107,-0.707107,1.0,0.0
1,938,51799,1788,70,5.000000e-01,-0.866025,0.918958,0.394356,-0.707107,-0.707107,...,5f7ca800fcb9368f78e3740cb68a4c4ebc62b005cd15cf...,unknown_city,6.123234e-17,1.0,-0.612106,0.790776,0.707107,-0.707107,1.0,0.0
2,938,34204,1918,103,-1.836970e-16,-1.000000,-0.612106,0.790776,-0.965926,-0.258819,...,5f7ca800fcb9368f78e3740cb68a4c4ebc62b005cd15cf...,unknown_city,6.123234e-17,1.0,-0.612106,0.790776,0.707107,-0.707107,1.0,0.0
3,938,13009,4805,208,5.000000e-01,-0.866025,-0.954139,-0.299363,-0.258819,0.965926,...,5f7ca800fcb9368f78e3740cb68a4c4ebc62b005cd15cf...,unknown_city,6.123234e-17,1.0,-0.612106,0.790776,0.707107,-0.707107,1.0,0.0
4,938,131005,3584,70,-1.836970e-16,-1.000000,0.347305,0.937752,0.707107,-0.707107,...,5f7ca800fcb9368f78e3740cb68a4c4ebc62b005cd15cf...,unknown_city,6.123234e-17,1.0,-0.612106,0.790776,0.707107,-0.707107,1.0,0.0


1) CatBoostRanker

In [78]:
train_cbm = train_df.copy()
val_cbm = val_df.copy()

for col in cat_cols:
    train_cbm[col] = train_cbm[col].astype(str)
    val_cbm[col] = val_cbm[col].astype(str)

train_pool = Pool(
    data=train_cbm[feature_cols],
    label=train_cbm['relevance'],
    group_id=train_cbm['user_id'],
    cat_features=cat_cols
)
val_pool = Pool(
    data=val_cbm[feature_cols],
    label=val_cbm['relevance'],
    group_id=val_cbm['user_id'],
    cat_features=cat_cols
)

: 

In [ ]:
cbm = CatBoostRanker(
    max_depth=8,
    loss='YetiRank',
    eval_metric='NDCG:top=10',
    iterations=300,
    early_stopping_rounds=50,
    random_state=42,
    verbose=100
)

gc.collect()
cbm.train(train_pool, eval_set=[val_pool])

In [ ]:
importance = cbm.get_feature_importance(type='PredictionValuesChange')

imp_df = pd.DataFrame({
    'feature': feature_cols,
    'importance': importance
})

imp_df.sort_values('importance', acsending=True)

2) LightGBM

In [ ]:
train_lgb = train_df.copy()
val_lgb = val_df.copy()
train_group = train_lgb.groupby('user_id').size().values()
val_group = val_lgb.groupby('user_id').size().values()

for col in cat_cols:
    train_lgb[col] = train_lgb[col].astype('category')
    val_lgb[col] = val_lgb[col].astype('category')

train_dataset = lgb.Dataset(
    train_lgb[feature_cols],
    label=train_lgb['relevance'],
    group=train_group,
    cat_features=cat_cols
)

val_dataset = lgb.Dataset(
    val_lgb[feature_cols],
    label=val_lgb['relevance'],
    group=val_group,
    cat_features=cat_cols
)

In [ ]:
lgb_params = {
    'objective': 'lambdarank',
    'metric': 'ndcg',
    'ndcg_eval_at': [10],
    'learning_rate': 0.05,
    'max_depth': 8,
    'num_leaves': 32,
    'l2_leaf_reg': 6,
    'l1_leaf_reg': 0.1,
    'min_data_in_leaf': 100,
    'verbose': -1,
    'seed': 42
}

lgb_model = lgb.train(
    lgb_params,
    train_dataset,
    num_boost_round=300,
    valid_sets=[val_dataset],
    valid_names=['val'],
    callbacks=[
        lgb.early_stopping(stopping_rounds=50),
        lgb.log_evaluation(period=100)
    ]
)

In [ ]:
lgb_imp = pd.DataFrame({
    'feature': feature_cols,
    'importance': np.log(lgb_model.feature_importance(importance_type='gain'))
})
print("LightGBM feature importance:")
lgb_imp.sort_values('importance', ascending=False)

Generate submission

In [ ]:
cands = candidates.copy() 
cnads = add_features(cands)

In [ ]:
cands_cbm = cands.copy()
for col in cat_cols:
    cands_cbm[col] = cands_cbm[col].astype(str)
score_ccbm = cbm.predict(cands_cbm[feature_cols])

cands_lgb = cands.copy()
for col in cat_cols:
    cands_cbm[col] = cands_lgb[col].astype('category')
score_lgb = lgb_model.predict(cands_lgb[feature_cols])

cands['score_cbm'] = score_ccbm
cands['score_lgb'] = cands_lgb

eps = 1e-8
for col in ['score_cbm', 'score_lgb']:
    cands[col+'_norm'] = cands.groupby('user_id')[col].transform(lambda x: (x - x.min()) / (x.max() - x.min() + eps))

w = 0.8
cands['score_blend'] = (
    w*cands['score_cbm_norm'] + (1-w)*cands['score_lgb_norm']
)

In [ ]:
target_users = set(sample_sub['user_id'].unique())
submission = cands[cands['user_id'].isin(target_users)][['user_id', 'edition_id', 'score_blend']]
submission = submission.sort_values(['user_id', 'score_blend'], ascending=[True, False])
submission = submission.groupby('user_id').head(10).reset_index(drop=True)
submission['rank'] = submission.groupby('user_id').cumcount() + 1
submission = submission[['user_id', 'edition_id', 'rank']]

submission.head(15)